# Eurostat & OECD Transport Data Analysis

This notebook analyzes Eurostat and OECD transport datasets with optimized functions for handling official statistical data formats.

## Features
- Automatic detection and loading of Eurostat/OECD CSV formats
- Multi-file batch processing
- Flag and metadata interpretation
- Time series analysis and visualization
- Cross-country and cross-dataset comparisons
- Memory-efficient data handling

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
from datetime import datetime
from typing import Dict, List, Tuple

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}' if abs(x) > 0.01 else f'{x:.2e}')

# Plotting settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

warnings.filterwarnings('ignore')

print('Libraries imported successfully!')

## 2. Data Loading Functions

Custom functions to handle Eurostat and OECD data formats

In [ ]:
def load_eurostat_csv(filepath: str) -> pd.DataFrame:
    """
    Load Eurostat CSV file with proper data type handling and flag interpretation.
    
    Args:
        filepath: Path to the Eurostat CSV file
        
    Returns:
        pd.DataFrame: Loaded and processed dataframe
    """
    df = pd.read_csv(filepath, low_memory=False)
    
    # Convert TIME_PERIOD to datetime if it exists
    if 'TIME_PERIOD' in df.columns:
        df['TIME_PERIOD'] = pd.to_datetime(df['TIME_PERIOD'], format='%Y', errors='coerce')
    
    # Convert OBS_VALUE to numeric, coercing errors to NaN
    if 'OBS_VALUE' in df.columns:
        df['OBS_VALUE'] = pd.to_numeric(df['OBS_VALUE'], errors='coerce')
    
    # Optimize memory by converting object columns to category where appropriate
    for col in df.columns:
        if df[col].dtype == 'object' and col not in ['LAST UPDATE']:
            num_unique = df[col].nunique()
            if num_unique / len(df) < 0.5:  # If less than 50% unique values
                df[col] = df[col].astype('category')
    
    return df


def load_oecd_csv(filepath: str) -> pd.DataFrame:
    """
    Load OECD CSV file handling the double-header format.
    
    Args:
        filepath: Path to the OECD CSV file
        
    Returns:
        pd.DataFrame: Loaded and processed dataframe
    """
    # Read the file to detect format
    with open(filepath, 'r') as f:
        first_line = f.readline()
    
    # OECD files often have metadata in first row
    if 'STRUCTURE' in first_line:
        # Skip the metadata row and use the actual header
        df = pd.read_csv(filepath, skiprows=1, low_memory=False)
    else:
        df = pd.read_csv(filepath, low_memory=False)
    
    # Convert TIME_PERIOD to datetime if it exists
    if 'TIME_PERIOD' in df.columns:
        df['TIME_PERIOD'] = pd.to_datetime(df['TIME_PERIOD'], format='%Y', errors='coerce')
    elif 'Time period' in df.columns:
        df['TIME_PERIOD'] = pd.to_datetime(df['Time period'], format='%Y', errors='coerce')
    
    # Convert OBS_VALUE to numeric
    if 'OBS_VALUE' in df.columns:
        df['OBS_VALUE'] = pd.to_numeric(df['OBS_VALUE'], errors='coerce')
    elif 'Observation value' in df.columns:
        df['OBS_VALUE'] = pd.to_numeric(df['Observation value'], errors='coerce')
    
    # Optimize memory
    for col in df.columns:
        if df[col].dtype == 'object':
            num_unique = df[col].nunique()
            if num_unique / len(df) < 0.5:
                df[col] = df[col].astype('category')
    
    return df


def load_all_datasets(directory: str = '.') -> Dict[str, pd.DataFrame]:
    """
    Load all CSV files from a directory.
    
    Args:
        directory: Directory containing CSV files
        
    Returns:
        Dictionary mapping filenames to dataframes
    """
    datasets = {}
    csv_files = list(Path(directory).glob('*.csv'))
    
    print(f'Found {len(csv_files)} CSV files\n')
    
    for filepath in csv_files:
        filename = filepath.name
        print(f'Loading {filename}...', end=' ')
        
        try:
            if 'OECD' in filename:
                df = load_oecd_csv(str(filepath))
            else:
                df = load_eurostat_csv(str(filepath))
            
            datasets[filename] = df
            print(f'✓ Loaded {df.shape[0]:,} rows × {df.shape[1]} columns')
        except Exception as e:
            print(f'✗ Error: {str(e)}')
    
    return datasets


def interpret_flags(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add human-readable interpretation of OBS_FLAG values.
    
    Common Eurostat flags:
    - b: break in time series
    - e: estimated
    - p: provisional
    - u: low reliability
    - c: confidential
    - m: missing/not available
    """
    flag_meanings = {
        'b': 'Break in time series',
        'B': 'Break in time series',
        'e': 'Estimated',
        'p': 'Provisional',
        'P': 'Provisional',
        'u': 'Low reliability',
        'c': 'Confidential',
        'm': 'Missing/Not available',
        'd': 'Definition differs'
    }
    
    if 'OBS_FLAG' in df.columns:
        df['FLAG_MEANING'] = df['OBS_FLAG'].map(flag_meanings)
    
    if 'OBS_STATUS' in df.columns:
        df['STATUS_MEANING'] = df['OBS_STATUS'].map(flag_meanings)
    
    return df

print('Data loading functions defined successfully!')

## 2B. Enhanced Filtering & Export Functions

Functions for filtering datasets and exporting to Excel with formatting.

In [ ]:
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

def get_filterable_columns(df, max_unique=50, exclude_cols=None):
    """
    Automatically identify columns suitable for filtering.
    
    Args:
        df: DataFrame to analyze
        max_unique: Maximum number of unique values for a column to be filterable
        exclude_cols: List of columns to exclude
        
    Returns:
        Dictionary with column names and their unique values
    """
    if exclude_cols is None:
        exclude_cols = ['DATAFLOW', 'LAST UPDATE', 'TIME_PERIOD', 'OBS_VALUE', 
                       'OBS_FLAG', 'CONF_STATUS', 'FLAG_MEANING', 'STATUS_MEANING']
    
    filterable = {}
    
    for col in df.columns:
        if col in exclude_cols:
            continue
        
        n_unique = df[col].nunique()
        if 0 < n_unique <= max_unique:
            filterable[col] = {
                'unique_count': n_unique,
                'values': df[col].value_counts().to_dict()
            }
    
    return filterable


def show_filter_options(df, max_unique=50):
    """
    Display available filter options for each filterable column.
    
    Args:
        df: DataFrame to analyze
        max_unique: Maximum number of unique values to display
    """
    print("="*70)
    print("🔍 FILTERABLE COLUMNS")
    print("="*70)
    print(f"Total rows in dataset: {len(df):,}\n")
    
    filterable = get_filterable_columns(df, max_unique)
    
    if not filterable:
        print("No filterable columns found (all columns have >50 unique values)")
        return filterable
    
    for col, info in filterable.items():
        print(f"\n📁 {col} ({info['unique_count']} unique values)")
        for value, count in sorted(info['values'].items(), 
                                   key=lambda x: x[1], reverse=True)[:10]:
            pct = (count / len(df)) * 100
            print(f"   {str(value):<30} {count:>7,} rows ({pct:>5.1f}%)")
        
        if info['unique_count'] > 10:
            print(f"   ... and {info['unique_count'] - 10} more values")
    
    print("\n" + "="*70)
    return filterable


def validate_filters(df, filters):
    """
    Validate that filter columns and values exist in the dataset.
    
    Args:
        df: DataFrame to validate against
        filters: Dictionary of {column: [values]} to validate
        
    Returns:
        Tuple of (is_valid, error_messages)
    """
    errors = []
    
    for col, values in filters.items():
        # Check if column exists
        if col not in df.columns:
            errors.append(f"❌ Column '{col}' not found in dataset")
            errors.append(f"   Available columns: {', '.join(df.columns[:10])}...")
            continue
        
        # Check if values exist
        valid_values = df[col].unique()
        invalid_values = [v for v in values if v not in valid_values]
        
        if invalid_values:
            errors.append(f"❌ Invalid values in column '{col}': {invalid_values}")
            errors.append(f"   Valid values: {list(valid_values)[:10]}...")
    
    is_valid = len(errors) == 0
    return is_valid, errors


def apply_filters(df, filters, verbose=True):
    """
    Apply filters to a DataFrame with validation and feedback.
    
    Args:
        df: DataFrame to filter
        filters: Dictionary of {column: [values]} to filter
        verbose: Print filtering progress
        
    Returns:
        Filtered DataFrame
    """
    # Validate filters first
    is_valid, errors = validate_filters(df, filters)
    
    if not is_valid:
        print("⚠️  FILTER VALIDATION ERRORS:\n")
        for error in errors:
            print(error)
        raise ValueError("Filter validation failed. Please fix the errors above.")
    
    # Apply filters
    df_filtered = df.copy()
    original_rows = len(df_filtered)
    
    if verbose:
        print("="*70)
        print("🔍 APPLYING FILTERS")
        print("="*70)
    
    for col, values in filters.items():
        if col in df_filtered.columns:
            before = len(df_filtered)
            df_filtered = df_filtered[df_filtered[col].isin(values)]
            after = len(df_filtered)
            
            if verbose:
                print(f"✓ {col}: {values}")
                print(f"  Rows: {before:,} → {after:,} (removed {before-after:,})")
    
    if verbose:
        print(f"\n📊 FINAL RESULT:")
        print(f"  Original rows: {original_rows:,}")
        print(f"  Filtered rows: {len(df_filtered):,}")
        print(f"  Removed: {original_rows - len(df_filtered):,} rows "
              f"({(original_rows - len(df_filtered))/original_rows*100:.1f}%)")
        print("="*70)
    
    return df_filtered


def export_to_xlsx(df, filename, sheet_name='Data', format_headers=True):
    """
    Export DataFrame to XLSX with formatting.
    
    Args:
        df: DataFrame to export
        filename: Output filename (should end in .xlsx)
        sheet_name: Name of the sheet
        format_headers: Apply formatting to headers
    """
    if not filename.endswith('.xlsx'):
        filename += '.xlsx'
    
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=False)
        
        if format_headers:
            worksheet = writer.sheets[sheet_name]
            
            # Format header row
            header_fill = PatternFill(start_color='366092', end_color='366092', 
                                     fill_type='solid')
            header_font = Font(color='FFFFFF', bold=True)
            
            for col_num, column in enumerate(df.columns, 1):
                cell = worksheet.cell(row=1, column=col_num)
                cell.fill = header_fill
                cell.font = header_font
                cell.alignment = Alignment(horizontal='center', vertical='center')
                
                # Auto-adjust column width
                max_length = max(
                    len(str(column)),
                    df[column].astype(str).str.len().max() if len(df) > 0 else 0
                )
                adjusted_width = min(max_length + 2, 50)
                worksheet.column_dimensions[get_column_letter(col_num)].width = adjusted_width
            
            # Freeze header row
            worksheet.freeze_panes = 'A2'
    
    print(f"✓ Exported to: {filename}")


print('✓ Enhanced filtering and export functions loaded successfully!')

## 3. Load Datasets

In [ ]:
# Load all CSV files in the current directory
datasets = load_all_datasets()

print(f'\n{"="*60}')
print(f'Successfully loaded {len(datasets)} datasets')
print(f'{"="*60}')

## 4. Dataset Overview

In [ ]:
# Overview of all datasets
overview_data = []

for name, dataset in datasets.items():  # ← Changed variable name
    overview_data.append({
        'Dataset': name[:50],
        'Rows': f'{dataset.shape[0]:,}',  # ← Use 'dataset'
        'Columns': dataset.shape[1],      # ← Use 'dataset'
        'Memory (MB)': f'{dataset.memory_usage(deep=True).sum() / 1024**2:.2f}',
        'Time Range': f"{dataset['TIME_PERIOD'].min().year if 'TIME_PERIOD' in dataset.columns and not dataset['TIME_PERIOD'].isna().all() else 'N/A'} - {dataset['TIME_PERIOD'].max().year if 'TIME_PERIOD' in dataset.columns and not dataset['TIME_PERIOD'].isna().all() else 'N/A'}",
        'Countries': dataset['geo'].nunique() if 'geo' in dataset.columns else (dataset['REF_AREA'].nunique() if 'REF_AREA' in dataset.columns else 'N/A')
    })

overview_df = pd.DataFrame(overview_data)
display(overview_df)

## 5. Detailed Analysis of Individual Datasets

In [ ]:
# Select a dataset for detailed analysis (change as needed)
# By default, select the first dataset
selected_dataset_name = list(datasets.keys())[2]
df = datasets[selected_dataset_name].copy()

# Add flag interpretations
df = interpret_flags(df)

print(f'Analyzing: {selected_dataset_name}')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n')

In [ ]:
# Quick Overview
print('First 5 rows:')
display(df.head())

print('\nColumn Info:')
df.info()

print('\nMissing Values:')
missing = df.isnull().sum()
if missing.sum() > 0:
    display(missing[missing > 0])
else:
    print('No missing values!')

## 6. Filter Dataset (Optional)

Apply filters to focus on a subset of the data. This is useful when you want to analyze specific countries, time periods, or categories.

In [ ]:
# Step 1: See what columns you can filter and their available values
show_filter_options(df)

In [ ]:
# Step 2: Define your filters based on the columns shown above
# Customize these filters based on what you saw in show_filter_options()

# Example filters - EDIT THESE based on your dataset:
my_filters = {
    # 'geo': ['BE', 'NL', 'DE'],     # Countries
    # 'tra_cov': ['TOTAL'],           # Transport coverage
    # 'unit': ['MIO_TKM'],            # Unit of measurement
}

# Option A: Apply filters (if you defined any above)
if my_filters:
    df = apply_filters(df, my_filters, verbose=True)
else:
    print("No filters defined. Using full dataset for analysis.")
    print(f"Current dataset has {len(df):,} rows")

In [ ]:
print("🔍 DETECTING KEY COLUMNS")

# Geographic column
geo_col = None
for col in ['geo', 'REF_AREA', 'LOCATION', 'Country', 'COUNTRY']:
    if col in df.columns:
        geo_col = col
        print(f"✓ Geographic column: {geo_col} ({df[geo_col].nunique()} unique values)")
        break

if not geo_col:
    print("ℹ️  No geographic column found")

# Time column
time_col = None
for col in ['TIME_PERIOD', 'Year', 'Date', 'TIME', 'YEAR']:
    if col in df.columns:
        time_col = col
        if df[time_col].dtype == 'datetime64[ns]':
            print(f"✓ Time column: {time_col} ({df[time_col].min().year} - {df[time_col].max().year})")
        else:
            print(f"✓ Time column: {time_col}")
        break

if not time_col:
    print("ℹ️  No time column found")

# Value column
value_col = None
for col in ['OBS_VALUE', 'Value', 'Observation', 'VALUE']:
    if col in df.columns:
        value_col = col
        print(f"✓ Value column: {value_col} ({df[value_col].notna().sum():,} non-null values)")
        break

if not value_col:
    print("ℹ️  No value column found")


## 9. Category Analysis

Analyze different categories/dimensions in the dataset

In [ ]:
def detect_key_columns(df):
    """
    Automatically detect geographic, time, and value columns in the dataset.
    
    Returns:
        Tuple of (geo_col, time_col, value_col)
    """
    # Detect geographic column
    geo_col = None
    for col in ['geo', 'REF_AREA', 'LOCATION', 'Country', 'COUNTRY']:
        if col in df.columns:
            geo_col = col
            break
    
    # Detect time column
    time_col = None
    for col in ['TIME_PERIOD', 'Year', 'Date', 'TIME', 'YEAR']:
        if col in df.columns:
            time_col = col
            break
    
    # Detect value column
    value_col = None
    for col in ['OBS_VALUE', 'Value', 'Observation', 'VALUE']:
        if col in df.columns:
            value_col = col
            break
    
    return geo_col, time_col, value_col


def create_analysis_summary(df, geo_col=None, time_col=None, value_col=None):
    """
    Create comprehensive analysis summary DataFrames.
    
    Returns:
        Dictionary of analysis dataframes
    """
    analysis = {}
    
    # Auto-detect columns if not provided
    if geo_col is None or time_col is None or value_col is None:
        geo_col, time_col, value_col = detect_key_columns(df)
    
    # Overall statistics
    if value_col and value_col in df.columns:
        stats = df[value_col].describe().to_frame('Value')
        stats.loc['missing'] = df[value_col].isna().sum()
        stats.loc['missing_pct'] = (df[value_col].isna().sum() / len(df)) * 100
        analysis['overall_stats'] = stats
    
    # Time series analysis
    if time_col and value_col and time_col in df.columns and value_col in df.columns:
        ts = df.groupby(time_col)[value_col].agg([
            ('count', 'count'),
            ('sum', 'sum'),
            ('mean', 'mean'),
            ('min', 'min'),
            ('max', 'max')
        ]).round(2)
        ts.columns = ['Count', 'Total', 'Average', 'Min', 'Max']
        analysis['time_series'] = ts
    
    # Geographic analysis
    if geo_col and value_col and geo_col in df.columns and value_col in df.columns:
        geo = df.groupby(geo_col)[value_col].agg([
            ('count', 'count'),
            ('sum', 'sum'),
            ('mean', 'mean'),
            ('min', 'min'),
            ('max', 'max')
        ]).round(2)
        geo.columns = ['Count', 'Total', 'Average', 'Min', 'Max']
        geo = geo.sort_values('Total', ascending=False)
        analysis['geographic'] = geo
    
    # Category analysis (for filterable categorical columns)
    filterable = get_filterable_columns(df, max_unique=20)
    for cat_col in filterable.keys():
        if cat_col not in [geo_col, time_col] and value_col and value_col in df.columns:
            cat = df.groupby(cat_col)[value_col].agg([
                ('count', 'count'),
                ('sum', 'sum'),
                ('mean', 'mean')
            ]).round(2)
            cat.columns = ['Count', 'Total', 'Average']
            cat = cat.sort_values('Total', ascending=False)
            analysis[f'by_{cat_col}'] = cat
    
    return analysis


def export_analysis_to_xlsx(df, filename, filters_applied=None, 
                            include_raw_data=True, include_analysis=True):
    """
    Export filtered data with comprehensive analysis to multi-sheet Excel file.
    
    Args:
        df: Filtered DataFrame to export
        filename: Output filename
        filters_applied: Dictionary of filters that were applied (for metadata)
        include_raw_data: Include the raw filtered data
        include_analysis: Include analysis sheets
    
    Returns:
        Path to created file
    """
    if not filename.endswith('.xlsx'):
        filename += '.xlsx'
    
    print("="*70)
    print("📊 EXPORTING DATA WITH ANALYSIS TO XLSX")
    print("="*70)
    
    # Detect key columns
    geo_col, time_col, value_col = detect_key_columns(df)
    
    print(f"\n🔍 Detected columns:")
    print(f"   Geographic: {geo_col if geo_col else 'Not found'}")
    print(f"   Time: {time_col if time_col else 'Not found'}")
    print(f"   Value: {value_col if value_col else 'Not found'}")
    
    # Create analysis
    if include_analysis:
        print(f"\n📈 Generating analysis...")
        analysis = create_analysis_summary(df, geo_col, time_col, value_col)
        print(f"   Created {len(analysis)} analysis tables")
    
    # Export to Excel
    print(f"\n💾 Writing to Excel...")
    
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        sheet_num = 1
        
        # Sheet 1: Metadata
        metadata = pd.DataFrame([
            {'Info': 'Export Date', 'Value': datetime.now().strftime('%Y-%m-%d %H:%M:%S')},
            {'Info': 'Total Rows', 'Value': len(df)},
            {'Info': 'Total Columns', 'Value': len(df.columns)},
            {'Info': 'Geographic Column', 'Value': geo_col if geo_col else 'N/A'},
            {'Info': 'Time Column', 'Value': time_col if time_col else 'N/A'},
            {'Info': 'Value Column', 'Value': value_col if value_col else 'N/A'},
        ])
        
        if filters_applied:
            metadata = pd.concat([metadata, pd.DataFrame([
                {'Info': 'Filters Applied', 'Value': 'Yes'},
                {'Info': 'Filter Details', 'Value': str(filters_applied)}
            ])])
        
        metadata.to_excel(writer, sheet_name='Metadata', index=False)
        print(f"   ✓ Sheet {sheet_num}: Metadata")
        sheet_num += 1
        
        # Sheet 2: Raw filtered data
        if include_raw_data:
            df.to_excel(writer, sheet_name='Filtered Data', index=False)
            print(f"   ✓ Sheet {sheet_num}: Filtered Data ({len(df):,} rows)")
            sheet_num += 1
        
        # Analysis sheets
        if include_analysis:
            for analysis_name, analysis_df in analysis.items():
                sheet_name = analysis_name.replace('_', ' ').title()
                sheet_name = sheet_name[:31]  # Excel sheet name limit
                analysis_df.to_excel(writer, sheet_name=sheet_name)
                print(f"   ✓ Sheet {sheet_num}: {sheet_name}")
                sheet_num += 1
        
        # Format all sheets
        header_fill = PatternFill(start_color='366092', end_color='366092', fill_type='solid')
        header_font = Font(color='FFFFFF', bold=True)
        
        for sheet_name in writer.sheets:
            worksheet = writer.sheets[sheet_name]
            
            # Format header row
            for cell in worksheet[1]:
                if cell.value:
                    cell.fill = header_fill
                    cell.font = header_font
                    cell.alignment = Alignment(horizontal='center', vertical='center')
            
            # Freeze panes
            worksheet.freeze_panes = 'A2'
            
            # Auto-adjust column widths
            for column in worksheet.columns:
                max_length = 0
                column_letter = column[0].column_letter
                for cell in column:
                    try:
                        if cell.value:
                            max_length = max(max_length, len(str(cell.value)))
                    except:
                        pass
                adjusted_width = min(max_length + 2, 50)
                worksheet.column_dimensions[column_letter].width = adjusted_width
    
    print(f"\n✅ EXPORT COMPLETE!")
    print(f"   File: {filename}")
    print(f"   Total sheets: {sheet_num - 1}")
    print("="*70)
    
    return filename


print('✓ Analysis and export functions loaded successfully!')

## 10. Cross-Dataset Comparison

Compare trends across multiple datasets

## Descriptive Analysis

## 11. Legacy CSV Export Functions

(Deprecated: Use export_analysis_to_xlsx() instead)

In [ ]:
# Check data quality
print('Data Quality Report:\n')

# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
}).sort_values('Missing Count', ascending=False)

print('Missing Values:')
display(missing_df[missing_df['Missing Count'] > 0])

# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f'\nDuplicate rows: {duplicates:,}')

# Flag distribution (if applicable)
if 'OBS_FLAG' in df.columns:
    print('\nObservation Flags Distribution:')
    flag_counts = df['OBS_FLAG'].value_counts(dropna=False)
    if 'FLAG_MEANING' in df.columns:
        # Create meanings dictionary including NaN
        meanings = df.groupby('OBS_FLAG')['FLAG_MEANING'].first().to_dict()
        meanings[np.nan] = 'No flag'  # Handle NaN values
        
        flag_summary = pd.DataFrame({
            'Flag': flag_counts.index,
            'Count': flag_counts.values,
            'Percentage': (flag_counts.values / len(df)) * 100,
            'Meaning': [meanings.get(flag, 'Unknown') for flag in flag_counts.index]
        })
    else:
        flag_summary = pd.DataFrame({
            'Flag': flag_counts.index,
            'Count': flag_counts.values,
            'Percentage': (flag_counts.values / len(df)) * 100
        })
    display(flag_summary)

## 12. Summary Report

In [ ]:
# Time Series by Country
if time_col and value_col and geo_col and df[geo_col].nunique() > 1:
    print("\n📊 TIME SERIES BY COUNTRY")
    
    pivot = df.pivot_table(
        values=value_col,
        index=time_col,
        columns=geo_col,
        aggfunc='sum',
        fill_value=0
    )
    
    fig, ax = plt.subplots(figsize=(14, 8))
    
    for country in pivot.columns:
        pivot[country].plot(ax=ax, marker='o', linewidth=2, label=country, alpha=0.8)
    
    ax.set_title('Time Series Comparison by Country', fontsize=14, fontweight='bold')
    ax.set_ylabel('Value')
    ax.legend(title='Country', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 13. Next Steps & Recommendations

Based on this analysis, consider:

1. **Data Quality Improvements**:
   - Handle missing data and flags appropriately
   - Investigate breaks in time series
   - Validate estimated and provisional values

2. **Further Analysis**:
   - Perform statistical tests (e.g., trend analysis, seasonality)
   - Build forecasting models
   - Compare performance across countries/regions
   - Analyze relationships between different transport modes

3. **Visualization**:
   - Create interactive dashboards (e.g., using Plotly)
   - Generate maps for geographical analysis
   - Build comparison reports

4. **Data Integration**:
   - Merge related datasets for comprehensive analysis
   - Add external data sources (e.g., economic indicators)
   - Create derived metrics and indicators

## 7. Time Series Analysis

In [ ]:
# Time series analysis
if 'TIME_PERIOD' in df.columns and 'OBS_VALUE' in df.columns:
    print('Time Series Analysis:\n')
    
    # Overall trend
    ts_data = df.groupby('TIME_PERIOD')['OBS_VALUE'].agg(['mean', 'sum', 'count'])
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    
    # Mean value over time
    axes[0].plot(ts_data.index, ts_data['mean'], marker='o', linewidth=2, markersize=4)
    axes[0].set_xlabel('Year')
    axes[0].set_ylabel('Mean Value')
    axes[0].set_title('Mean Value Over Time')
    axes[0].grid(True, alpha=0.3)
    
    # Total value over time
    axes[1].plot(ts_data.index, ts_data['sum'], marker='o', linewidth=2, markersize=4, color='green')
    axes[1].set_xlabel('Year')
    axes[1].set_ylabel('Total Value')
    axes[1].set_title('Total Value Over Time')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Calculate year-over-year growth
    ts_data['yoy_growth'] = ts_data['sum'].pct_change() * 100
    print('Year-over-Year Growth Rate (%):')
    print(ts_data[['sum', 'yoy_growth']].tail(10))
else:
    print('Time period data not available for time series analysis.')

In [ ]:
# Time series by country (for selected major countries)
if 'TIME_PERIOD' in df.columns and geo_col and 'OBS_VALUE' in df.columns:
    print('Time Series Comparison by Country:\n')
    
    # Select top countries by total value
    top_countries = df.groupby(geo_col)['OBS_VALUE'].sum().nlargest(6).index
    
    fig, ax = plt.subplots(figsize=(14, 7))
    
    for country in top_countries:
        country_data = df[df[geo_col] == country].groupby('TIME_PERIOD')['OBS_VALUE'].sum()
        ax.plot(country_data.index, country_data.values, marker='o', linewidth=2, 
                markersize=4, label=country, alpha=0.8)
    
    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel('Value', fontsize=12)
    ax.set_title('Time Series Comparison - Top 6 Countries', fontsize=14, fontweight='bold')
    ax.legend(loc='best', framealpha=0.9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 8. Category Analysis

Analyze different categories/dimensions in the dataset

In [ ]:
# Identify categorical columns (excluding metadata columns)
exclude_cols = ['DATAFLOW', 'LAST UPDATE', 'TIME_PERIOD', 'OBS_VALUE', 'OBS_FLAG', 
                'CONF_STATUS', 'FLAG_MEANING', 'STATUS_MEANING', geo_col]
categorical_cols = [col for col in df.select_dtypes(include=['object', 'category']).columns 
                   if col not in exclude_cols and df[col].nunique() < 50]

if categorical_cols:
    print(f'Found {len(categorical_cols)} categorical dimension(s) for analysis:\n')
    
    for col in categorical_cols:
        print(f'\n{"="*60}')
        print(f'Category: {col}')
        print(f'{"="*60}')
        
        # Value counts
        value_counts = df[col].value_counts()
        print(f'\nUnique values: {len(value_counts)}')
        display(value_counts.to_frame('Count'))
        
        # Analysis by category if we have numerical values
        if 'OBS_VALUE' in df.columns:
            category_stats = df.groupby(col)['OBS_VALUE'].agg(['count', 'mean', 'sum']).round(2)
            category_stats.columns = ['Observations', 'Mean', 'Total']
            category_stats = category_stats.sort_values('Total', ascending=False)
            
            print(f'\nStatistics by {col}:')
            display(category_stats)
            
            # Visualization
            if len(value_counts) <= 20:  # Only plot if not too many categories
                fig, axes = plt.subplots(1, 2, figsize=(16, 6))
                
                # Bar plot of totals
                category_stats['Total'].plot(kind='barh', ax=axes[0], color='steelblue')
                axes[0].set_xlabel('Total Value')
                axes[0].set_ylabel(col)
                axes[0].set_title(f'Total Value by {col}')
                axes[0].grid(True, alpha=0.3)
                
                # Bar plot of means
                category_stats['Mean'].plot(kind='barh', ax=axes[1], color='coral')
                axes[1].set_xlabel('Mean Value')
                axes[1].set_ylabel(col)
                axes[1].set_title(f'Mean Value by {col}')
                axes[1].grid(True, alpha=0.3)
                
                plt.tight_layout()
                plt.show()
else:
    print('No categorical dimensions found for analysis.')

## 9. Cross-Dataset Comparison

Compare trends across multiple datasets

In [ ]:
# Compare time series across all datasets
if len(datasets) > 1:
    print('Cross-Dataset Comparison:\n')
    
    fig, ax = plt.subplots(figsize=(14, 8))
    
    comparison_data = []
    
    for name, dataset in datasets.items():
        if 'TIME_PERIOD' in dataset.columns and 'OBS_VALUE' in dataset.columns:
            ts = dataset.groupby('TIME_PERIOD')['OBS_VALUE'].sum()
            
            # Normalize for comparison (to 0-100 scale)
            if ts.max() > 0:
                ts_normalized = (ts / ts.max()) * 100
                ax.plot(ts_normalized.index, ts_normalized.values, 
                       marker='o', linewidth=2, markersize=3, 
                       label=name[:40], alpha=0.7)  # Truncate long names
            
            comparison_data.append({
                'Dataset': name[:40],
                'Time Range': f"{ts.index.min().year} - {ts.index.max().year}",
                'Data Points': len(ts),
                'Total Value': f'{ts.sum():,.0f}',
                'Mean Value': f'{ts.mean():,.2f}'
            })
    
    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel('Normalized Value (% of maximum)', fontsize=12)
    ax.set_title('Cross-Dataset Time Series Comparison (Normalized)', fontsize=14, fontweight='bold')
    ax.legend(loc='best', framealpha=0.9, fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Summary table
    print('\nDataset Comparison Summary:')
    comparison_df = pd.DataFrame(comparison_data)
    display(comparison_df)
else:
    print('Only one dataset loaded. Load multiple datasets for cross-comparison.')

## 10. Data Export Functions

In [ ]:
def export_analysis_results(df: pd.DataFrame, geo_col: str = None, prefix: str = 'analysis'):
    """
    Export key analysis results to CSV files.
    
    Args:
        df: DataFrame to analyze
        geo_col: Name of geography column
        prefix: Prefix for output filenames
    """
    # Time series summary
    if 'TIME_PERIOD' in df.columns and 'OBS_VALUE' in df.columns:
        ts_summary = df.groupby('TIME_PERIOD')['OBS_VALUE'].agg(['count', 'mean', 'sum'])
        ts_summary.to_csv(f'{prefix}_time_series.csv')
        print(f'✓ Exported: {prefix}_time_series.csv')
    
    # Geographic summary
    if geo_col and 'OBS_VALUE' in df.columns:
        geo_summary = df.groupby(geo_col)['OBS_VALUE'].agg(['count', 'mean', 'sum'])
        geo_summary.to_csv(f'{prefix}_geographic.csv')
        print(f'✓ Exported: {prefix}_geographic.csv')
    
    # Cleaned full dataset
    df_clean = df.dropna(subset=['OBS_VALUE'])
    df_clean.to_csv(f'{prefix}_cleaned.csv', index=False)
    print(f'✓ Exported: {prefix}_cleaned.csv')

# Example usage (uncomment to use):
# export_analysis_results(df, geo_col, 'transport_analysis')

print('Export functions ready to use!')

## 11. Summary Report

In [ ]:
# Generate comprehensive summary report
print('='*70)
print('TRANSPORT DATA ANALYSIS SUMMARY REPORT')
print('='*70)

print(f'\n📊 DATASETS LOADED: {len(datasets)}')
print(f'\nTotal observations across all datasets: {sum(d.shape[0] for d in datasets.values()):,}')
print(f'Total memory usage: {sum(d.memory_usage(deep=True).sum() for d in datasets.values()) / 1024**2:.2f} MB')

if datasets:
    print(f'\n📁 DATASET DETAILS:')
    for name, dataset in datasets.items():
        print(f'\n  • {name}')
        print(f'    - Rows: {dataset.shape[0]:,}')
        print(f'    - Columns: {dataset.shape[1]}')
        
        if 'TIME_PERIOD' in dataset.columns:
            valid_dates = dataset['TIME_PERIOD'].dropna()
            if len(valid_dates) > 0:
                print(f'    - Time range: {valid_dates.min().year} - {valid_dates.max().year}')
        
        geo_column = 'geo' if 'geo' in dataset.columns else ('REF_AREA' if 'REF_AREA' in dataset.columns else None)
        if geo_column:
            print(f'    - Countries/Regions: {dataset[geo_column].nunique()}')
        
        if 'OBS_VALUE' in dataset.columns:
            valid_obs = dataset['OBS_VALUE'].dropna()
            if len(valid_obs) > 0:
                print(f'    - Valid observations: {len(valid_obs):,}')
                print(f'    - Value range: {valid_obs.min():,.0f} - {valid_obs.max():,.0f}')

print(f'\n{"="*70}')
print('✓ Analysis complete!')
print(f'{"="*70}')

## 12. Next Steps & Recommendations

Based on this analysis, consider:

1. **Data Quality Improvements**:
   - Handle missing data and flags appropriately
   - Investigate breaks in time series
   - Validate estimated and provisional values

2. **Further Analysis**:
   - Perform statistical tests (e.g., trend analysis, seasonality)
   - Build forecasting models
   - Compare performance across countries/regions
   - Analyze relationships between different transport modes

3. **Visualization**:
   - Create interactive dashboards (e.g., using Plotly)
   - Generate maps for geographical analysis
   - Build comparison reports

4. **Data Integration**:
   - Merge related datasets for comprehensive analysis
   - Add external data sources (e.g., economic indicators)
   - Create derived metrics and indicators